# Notebook A — Feature Extraction

**Purpose:** Extract three feature representations from the OAHS dataset
and cache them on Drive. This notebook runs **once**. Notebooks B and C
load the cached features instantly.

**Features extracted:**
| Feature | Shape | Method |
|---|---|---|
| DWT scalogram | (6, 94, 1) | db4, 5 levels, resampled to 94 frames |
| MFCC | (40, 94, 1) | 40 coeffs, hop=64 |
| Log-mel spectrogram | (64, 94, 1) | 64 mel bins, hop=64 |

**Shared properties:**
- All three features use **the same 94-frame time axis** so all model
  architectures work on every feature with no architecture changes
- All three are **z-score normalized** per-sample
- A single set of **5-fold StratifiedKFold indices** is saved alongside
  the features so all 22 experiments use identical splits

**Output location:** `/content/drive/MyDrive/Msc_ML_project/features_v1/`

**Files produced:**
- `oahs_dwt.npz`
- `oahs_mfcc.npz`
- `oahs_logmel.npz`
- `oahs_folds.npz` (the 5-fold splits)
- `oahs_files.npy` (recording filenames, for traceability)


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install --upgrade librosa resampy pywavelets --quiet

In [ ]:
import os, glob, time
import numpy as np
import librosa
import pywt
import matplotlib.pyplot as plt
from scipy.signal import resample
from sklearn.model_selection import StratifiedKFold

SEED = 42
np.random.seed(SEED)
print('Libraries imported.')

## 2. Configuration

Single source of truth. Notebooks B and C will read these exact same
values to ensure compatibility.

In [ ]:
# ---------- Paths ----------
INPUT_DIR  = '/content/drive/MyDrive/Msc_ML_project/dataset'
OUTPUT_DIR = '/content/drive/MyDrive/Msc_ML_project/features_v1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------- Audio ----------
SR        = 2000
DURATION  = 3.0
N_SAMPLES = int(SR * DURATION)   # 6000

# ---------- Common time axis ----------
# Every feature ends up with shape (rows, 94, 1) so all 7 model
# architectures work on every feature with no changes.
N_TIME = 94

# ---------- Log-Mel ----------
N_FFT      = 512
HOP_LENGTH = 64                   # → 94 frames over 3s @ 2000 Hz
WINDOW     = 'hann'
N_MELS     = 64
F_MIN      = 20
F_MAX      = 1000

# ---------- MFCC ----------
N_MFCC = 40

# ---------- DWT ----------
DWT_WAVELET   = 'db4'
DWT_LEVELS    = 5                 # → 6 coefficient arrays (cA5, cD5..cD1)
DWT_ROWS      = DWT_LEVELS + 1    # 6 rows in the scalogram

# ---------- Classes ----------
CLASSES   = ['AS', 'MR', 'MS', 'MVP', 'N']
LABEL_MAP = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

# ---------- Cross-validation ----------
N_FOLDS = 5

EPS = 1e-10

print('Configuration loaded.')
print(f'  Audio: SR={SR}, duration={DURATION}s, N_SAMPLES={N_SAMPLES}')
print(f'  Common time axis: {N_TIME} frames')
print(f'  DWT shape:   ({DWT_ROWS}, {N_TIME}, 1)')
print(f'  MFCC shape:  ({N_MFCC}, {N_TIME}, 1)')
print(f'  LogMel shape:({N_MELS}, {N_TIME}, 1)')

## 3. Audio preprocessing

Three steps for every wav: load+truncate/pad → RMS normalize → z-score normalize.
**No bandpass** — the mel filterbank (and the DWT decomposition) do their
own implicit frequency selection.

In [ ]:
def fix_len(x, n=N_SAMPLES):
    if len(x) < n:
        return librosa.util.fix_length(x, size=n)
    return x[:n]

def rms_normalize(x):
    return x / (np.sqrt(np.mean(x**2)) + EPS)

def zscore(x):
    return (x - np.mean(x)) / (np.std(x) + EPS)

def load_and_clean(path, sr=SR, duration=DURATION):
    x, _ = librosa.load(path, sr=sr, duration=duration, res_type='kaiser_fast')
    x = fix_len(x, N_SAMPLES)
    x = rms_normalize(x)
    x = zscore(x)
    return x.astype(np.float32)

print('Audio preprocessing functions defined.')

## 4. Feature extractors — DWT, MFCC, log-mel

Each returns a `(rows, N_TIME, 1)` float32 array, z-score normalized.

In [ ]:
def wave_to_logmel(x, sr=SR):
    """Log-mel spectrogram. Shape: (N_MELS, N_TIME, 1)."""
    S = librosa.feature.melspectrogram(
        y=x, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX, window=WINDOW, power=2.0)
    S_db = librosa.power_to_db(S, ref=np.max)
    if S_db.shape[1] < N_TIME:
        pad = np.full((N_MELS, N_TIME - S_db.shape[1]), S_db.min(), dtype=np.float32)
        S_db = np.hstack([S_db, pad])
    else:
        S_db = S_db[:, :N_TIME]
    S_db = (S_db - np.mean(S_db)) / (np.std(S_db) + EPS)
    return S_db.astype(np.float32)[..., np.newaxis]


def wave_to_mfcc(x, sr=SR):
    """MFCC time-coefficient matrix. Shape: (N_MFCC, N_TIME, 1)."""
    M = librosa.feature.mfcc(
        y=x, sr=sr, n_mfcc=N_MFCC,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX, window=WINDOW)
    if M.shape[1] < N_TIME:
        pad = np.zeros((N_MFCC, N_TIME - M.shape[1]), dtype=np.float32)
        M = np.hstack([M, pad])
    else:
        M = M[:, :N_TIME]
    M = (M - np.mean(M)) / (np.std(M) + EPS)
    return M.astype(np.float32)[..., np.newaxis]


def wave_to_dwt_scalogram(x):
    """DWT scalogram. Each decomposition level is resampled to N_TIME
    so all rows share the same time axis. Shape: (DWT_ROWS, N_TIME, 1).

    Row 0 = coarsest approximation (cA at top level)
    Row 1..DWT_LEVELS = details from coarsest (cD_top) to finest (cD_1)
    """
    coeffs = pywt.wavedec(x, DWT_WAVELET, level=DWT_LEVELS)
    # coeffs = [cA_top, cD_top, cD_{top-1}, ..., cD_1]
    rows = []
    for c in coeffs:
        # Resample each coefficient array to N_TIME using FFT-based resampling
        # (works whether the coeff array is longer or shorter than N_TIME)
        if len(c) == N_TIME:
            r = c
        else:
            r = resample(c, N_TIME)
        rows.append(r.astype(np.float32))
    S = np.stack(rows, axis=0)            # (DWT_ROWS, N_TIME)
    S = (S - np.mean(S)) / (np.std(S) + EPS)
    return S.astype(np.float32)[..., np.newaxis]


print('Feature extractors defined.')
print('Quick shape sanity check:')
x_dummy = np.random.randn(N_SAMPLES).astype(np.float32)
print(f'  log-mel: {wave_to_logmel(x_dummy).shape}')
print(f'  MFCC   : {wave_to_mfcc(x_dummy).shape}')
print(f'  DWT    : {wave_to_dwt_scalogram(x_dummy).shape}')

## 5. Run extraction over the whole dataset

Single pass over all wavs — each file is read once and all three features
are computed before moving to the next file.

In [ ]:
def extract_all_features():
    X_dwt_list, X_mfcc_list, X_spec_list = [], [], []
    y_list, file_list = [], []

    t_start = time.time()
    for cls in CLASSES:
        folder = os.path.join(INPUT_DIR, cls)
        if not os.path.isdir(folder):
            print(f'WARNING: missing folder {folder}')
            continue
        wavs = sorted(glob.glob(os.path.join(folder, '*.wav')))
        print(f'  {cls:>3}: {len(wavs)} files')
        for p in wavs:
            try:
                w = load_and_clean(p)
                X_dwt_list.append(wave_to_dwt_scalogram(w))
                X_mfcc_list.append(wave_to_mfcc(w))
                X_spec_list.append(wave_to_logmel(w))
                y_list.append(LABEL_MAP[cls])
                file_list.append(f'{cls}/{os.path.basename(p)}')
            except Exception as e:
                print(f'  ! failed {p}: {e}')

    X_dwt  = np.stack(X_dwt_list).astype(np.float32)
    X_mfcc = np.stack(X_mfcc_list).astype(np.float32)
    X_spec = np.stack(X_spec_list).astype(np.float32)
    y      = np.array(y_list,    dtype=np.int64)
    files  = np.array(file_list)

    elapsed = time.time() - t_start
    print(f'\nExtraction done in {elapsed:.1f}s')
    print(f'  X_dwt :  {X_dwt.shape}')
    print(f'  X_mfcc:  {X_mfcc.shape}')
    print(f'  X_spec:  {X_spec.shape}')
    print(f'  y     :  {y.shape}')
    print(f'  files :  {files.shape}')
    print(f'  Class distribution: '
          f'{ {CLASSES[i]: int((y == i).sum()) for i in range(NUM_CLASSES)} }')
    return X_dwt, X_mfcc, X_spec, y, files


X_dwt, X_mfcc, X_spec, y, files = extract_all_features()

## 6. Build the 5-fold splits

Single `StratifiedKFold(seed=42)` shared across all 22 experiments in
Notebook B. Saved as integer arrays of train/val indices.

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds_train_idx, folds_val_idx = [], []

# Use any of the X arrays — they all have the same number of recordings
for fold_i, (tr, va) in enumerate(skf.split(X_spec, y), start=1):
    folds_train_idx.append(tr.astype(np.int64))
    folds_val_idx.append(va.astype(np.int64))
    # Sanity: same class distribution in train/val
    val_dist = {CLASSES[i]: int((y[va] == i).sum()) for i in range(NUM_CLASSES)}
    print(f'  Fold {fold_i}: train={len(tr)}, val={len(va)}, val class dist={val_dist}')

# Store as object arrays since folds may have different lengths
folds_train_idx_arr = np.array(folds_train_idx, dtype=object)
folds_val_idx_arr   = np.array(folds_val_idx,   dtype=object)
print('\nFolds built.')

## 7. Save everything to Drive

In [ ]:
def save_feature(path, X, y, files):
    np.savez_compressed(path, X=X, y=y, files=files)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f'  Saved {os.path.basename(path)}  ({size_mb:.1f} MB)  shape={X.shape}')

print('Saving features...')
save_feature(os.path.join(OUTPUT_DIR, 'oahs_dwt.npz'),    X_dwt,  y, files)
save_feature(os.path.join(OUTPUT_DIR, 'oahs_mfcc.npz'),   X_mfcc, y, files)
save_feature(os.path.join(OUTPUT_DIR, 'oahs_logmel.npz'), X_spec, y, files)

print('\nSaving folds...')
folds_path = os.path.join(OUTPUT_DIR, 'oahs_folds.npz')
np.savez_compressed(
    folds_path,
    train_idx=folds_train_idx_arr,
    val_idx=folds_val_idx_arr,
    n_folds=N_FOLDS,
    seed=SEED,
)
print(f'  Saved {os.path.basename(folds_path)}  ({os.path.getsize(folds_path)/1024:.1f} KB)')

print('\nSaving filenames...')
files_path = os.path.join(OUTPUT_DIR, 'oahs_files.npy')
np.save(files_path, files)
print(f'  Saved {os.path.basename(files_path)}')

print('\nAll files in', OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    p = os.path.join(OUTPUT_DIR, f)
    print(f'  {f}  ({os.path.getsize(p)/1024:.1f} KB)')

## 8. Visual sanity check — all three features per class

If AS/MR/MVP look genuinely different in at least one feature, the
classifiers have something to work with. If they all look identical,
the project is doomed regardless of model choice.

In [ ]:
fig, axes = plt.subplots(3, NUM_CLASSES, figsize=(3 * NUM_CLASSES, 8))

for col, cls_idx in enumerate(range(NUM_CLASSES)):
    idx = np.where(y == cls_idx)[0][0]   # first example of this class
    axes[0, col].imshow(X_dwt[idx, ..., 0],  aspect='auto', origin='lower', cmap='cividis')
    axes[0, col].set_title(f'{CLASSES[cls_idx]} — DWT')
    axes[0, col].axis('off')

    axes[1, col].imshow(X_mfcc[idx, ..., 0], aspect='auto', origin='lower', cmap='viridis')
    axes[1, col].set_title(f'{CLASSES[cls_idx]} — MFCC')
    axes[1, col].axis('off')

    axes[2, col].imshow(X_spec[idx, ..., 0], aspect='auto', origin='lower', cmap='magma')
    axes[2, col].set_title(f'{CLASSES[cls_idx]} — Log-Mel')
    axes[2, col].axis('off')

plt.suptitle('Three feature representations per class\n'
             '(rows: DWT, MFCC, log-mel  |  cols: AS, MR, MS, MVP, N)', y=1.00)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'features_per_class.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved features_per_class.png')

## 9. Next step

This notebook is done. The cached files are now on Drive at:

```
/content/drive/MyDrive/Msc_ML_project/features_v1/
├── oahs_dwt.npz
├── oahs_mfcc.npz
├── oahs_logmel.npz
├── oahs_folds.npz
├── oahs_files.npy
└── features_per_class.png
```

**You do not need to run this notebook again** unless you change feature
parameters. The next notebook (B) will load all of this in seconds.

---

### Quick load example for Notebook B

```python
FEAT_DIR = '/content/drive/MyDrive/Msc_ML_project/features_v1'

dwt   = np.load(os.path.join(FEAT_DIR, 'oahs_dwt.npz'))
mfcc  = np.load(os.path.join(FEAT_DIR, 'oahs_mfcc.npz'))
spec  = np.load(os.path.join(FEAT_DIR, 'oahs_logmel.npz'))
folds = np.load(os.path.join(FEAT_DIR, 'oahs_folds.npz'), allow_pickle=True)

X_dwt, X_mfcc, X_spec = dwt['X'], mfcc['X'], spec['X']
y = dwt['y']
fold_train = list(folds['train_idx'])
fold_val   = list(folds['val_idx'])
```
